In [1]:
!hostname

cn021.delta.ncsa.illinois.edu


In [2]:
import multivelo as mv

/projects/bgdb/asachan/.conda/envs/multivelo/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/projects/bgdb/asachan/.conda/envs/multivelo/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/projects/bgdb/asachan/.conda/envs/multivelo/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/projects/bgdb/asachan/.conda/envs/multivelo/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/projects/bgdb/asachan/.conda/envs/multivelo/lib/python3.10/site-p

In [3]:
import jax
print(jax.devices())  # should show [CpuDevice(id=0)]
print(mv.__version__)

[CpuDevice(id=0)]
0.1.3


In [4]:
import os
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import anndata as ad

In [6]:
import matplotlib
matplotlib.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'figure.frameon': False,
    'font.size': 14,
    'figure.figsize': (6, 4),
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})
import matplotlib.pyplot as plt

scv.settings.verbosity = 3
scv.settings.presenter_view = True
# Skip scv.set_figure_params('scvelo') — causes matplotlib_inline version conflict
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
np.set_printoptions(suppress=True)

In [13]:
x_dr_file = '/work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/data/coord_rna.tsv.gz'
stream_obs_file = '/projects/bgdb/asachan/datasets/Bcell_in_vitro_human/adata_stream_obs.tsv.gz'
rna_adata_file = '/projects/bgdb/asachan/datasets/Bcell_in_vitro_human/bcell_rna.h5ad'
atac_adata_file = '/projects/bgdb/asachan/datasets/Bcell_in_vitro_human/bcell_atac.h5ad'

#### Checking loom files

In [10]:
loom_folder = '/work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related'

In [11]:
import loompy

ds = loompy.connect(
    os.path.join(loom_folder, "velocyto_day0_2/multiome_1st_donor_UPMC_day0_2.loom"),
    validate=False
)

# 1. Check cell barcodes
print("First 5 barcodes:", ds.ca["CellID"][:5])

# 2. Check matrix layers and shape
print("Layers:", list(ds.layers.keys()))
print("Shape (genes x cells):", ds.shape)

# 3. Check global/row/col attributes
print("Global attrs:", dict(ds.attrs))
print("Row attrs:", list(ds.ra.keys()))
print("Col attrs:", list(ds.ca.keys()))

ds.close()

First 5 barcodes: ['multiome_1st_donor_UPMC_day0_2:AAACGCGCAAGCTTATx'
 'multiome_1st_donor_UPMC_day0_2:AAACCGAAGTCATGCGx'
 'multiome_1st_donor_UPMC_day0_2:AAACATGCAGCTCAACx'
 'multiome_1st_donor_UPMC_day0_2:AAACGCGCACGCAACTx'
 'multiome_1st_donor_UPMC_day0_2:AAACCGAAGGGATGACx']
Layers: ['', 'ambiguous', 'spliced', 'unspliced']
Shape (genes x cells): (36601, 15285)
Global attrs: {'last_modified': '20240731T232513.499888Z'}
Row attrs: ['Accession', 'Chromosome', 'End', 'Gene', 'Start', 'Strand']
Col attrs: ['CellID', 'Clusters']


# Adapting code for multivelo 

In [14]:
BASE = "/work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2"

# Velocyto loom files (RNA spliced/unspliced)
LOOM_DIR = os.path.join(BASE, "velocity_related")
LOOM_FILES = {
    "day0_2": os.path.join(LOOM_DIR, "velocyto_day0_2/multiome_1st_donor_UPMC_day0_2.loom"),
    "day3_4": os.path.join(LOOM_DIR, "velocyto_day3_4/multiome_1st_donor_UPMC_day3_4.loom"), 
    "day5_6": os.path.join(LOOM_DIR, "velocyto_day5_6/multiome_1st_donor_UPMC_day5_6.loom"), 
}

# RNA h5ad has X_MultiVI in .obsm, cell type labels, UMAP, etc.
RNA_H5AD = "/projects/bgdb/asachan/datasets/Bcell_in_vitro_human/bcell_rna.h5ad"
ATAC_H5AD = "/projects/bgdb/asachan/datasets/Bcell_in_vitro_human/bcell_atac.h5ad"

# CellRanger ARC output directories (one per sample)
# Needed for: peak_annotation.tsv (atac_peak_annotation.tsv) + feature_linkage.bedpe
CELLRANGER_ARC_OUTS = {
    "day0_2": os.path.join(BASE, "/work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day0_2/outs"),
    "day3_4": os.path.join(BASE, "/work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day3_4/outs"),
    "day5_6": os.path.join(BASE, "/work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day5_6/outs"),
}

# Output directory
OUT_DIR = os.path.join(BASE, "multivelo_results")
os.makedirs(OUT_DIR, exist_ok=True)


### 2. Load & Merge Loom Files (RNA: spliced + unspliced)
#### Each loom has barcodes like `multiome_1st_donor_UPMC_day0_2:AAACGCGCAAGCTTATx`.
#### We need to harmonize these with the MuData barcodes.

In [15]:
import loompy
import anndata as ad
from scipy.sparse import csr_matrix

looms = {}
for tp, path in LOOM_FILES.items():
    print(f"\n--- Loading {tp} ---")
    ds = loompy.connect(path, validate=False)
    
    # Build AnnData manually
    adata = ad.AnnData(
        X=csr_matrix(ds.layers['spliced'][:, :].T), #transpose to genes x cells
        obs=pd.DataFrame(index=ds.ca['CellID']),
        var=pd.DataFrame(index=ds.ra['Gene']),
    )
    adata.layers['spliced'] = adata.X.copy()
    adata.layers['unspliced'] = csr_matrix(ds.layers['unspliced'][:, :].T)
    adata.layers['ambiguous'] = csr_matrix(ds.layers['ambiguous'][:, :].T)
    
    ds.close()
    
    adata.var_names_make_unique()
    adata.obs['timepoint'] = tp
    print(f"  Shape: {adata.shape}")
    print(f"  Layers: {list(adata.layers.keys())}")
    print(f"  Example barcodes: {adata.obs_names[:2].tolist()}")
    
    looms[tp] = adata


--- Loading day0_2 ---
  Shape: (15285, 36601)
  Layers: ['spliced', 'unspliced', 'ambiguous']
  Example barcodes: ['multiome_1st_donor_UPMC_day0_2:AAACGCGCAAGCTTATx', 'multiome_1st_donor_UPMC_day0_2:AAACCGAAGTCATGCGx']

--- Loading day3_4 ---
  Shape: (11127, 36601)
  Layers: ['spliced', 'unspliced', 'ambiguous']
  Example barcodes: ['multiome_1st_donor_UPMC_day3_4:AAACCAACACCTGCCTx', 'multiome_1st_donor_UPMC_day3_4:AAACATGCATAATCCGx']

--- Loading day5_6 ---
  Shape: (9894, 36601)
  Layers: ['spliced', 'unspliced', 'ambiguous']
  Example barcodes: ['multiome_1st_donor_UPMC_day5_6:AAAGCCGCACCTCACCx', 'multiome_1st_donor_UPMC_day5_6:AAACCGCGTGAGACTCx']


In [16]:
# Print raw barcodes from each source
for tp, adata in looms.items():
    print(f"\nLoom {tp}: {adata.obs_names[:3].tolist()}")
 
# Load exported RNA h5ad (contains X_MultiVI, cell types, UMAP)
adata_ref = sc.read_h5ad(RNA_H5AD)
print(f"\nRNA h5ad barcodes: {adata_ref.obs_names[:3].tolist()}")
print(f"RNA h5ad shape: {adata_ref.shape}")
print(f"RNA h5ad obsm keys: {list(adata_ref.obsm.keys())}")


Loom day0_2: ['multiome_1st_donor_UPMC_day0_2:AAACGCGCAAGCTTATx', 'multiome_1st_donor_UPMC_day0_2:AAACCGAAGTCATGCGx', 'multiome_1st_donor_UPMC_day0_2:AAACATGCAGCTCAACx']

Loom day3_4: ['multiome_1st_donor_UPMC_day3_4:AAACCAACACCTGCCTx', 'multiome_1st_donor_UPMC_day3_4:AAACATGCATAATCCGx', 'multiome_1st_donor_UPMC_day3_4:AAACCGCGTTTCCTCCx']

Loom day5_6: ['multiome_1st_donor_UPMC_day5_6:AAAGCCGCACCTCACCx', 'multiome_1st_donor_UPMC_day5_6:AAACCGCGTGAGACTCx', 'multiome_1st_donor_UPMC_day5_6:AAACGTACATGAGTTTx']

RNA h5ad barcodes: ['AAACAGCCAAGCCACT-3', 'AAACAGCCAAGGTGCA-1', 'AAACAGCCAAGTTATC-1']
RNA h5ad shape: (28494, 3018)
RNA h5ad obsm keys: ['DM_EigenVectors', 'X_MultiVI', 'X_joint_umap_features', 'X_pca', 'X_topic_compositions', 'X_umap', 'X_umap_features']


In [24]:
def harmonize_loom_barcodes(adata, timepoint):
    batch_map = {"day0_2": "1", "day3_4": "2", "day5_6": "3"}
    suffix = batch_map[timepoint]
    new_names = []
    for bc in adata.obs_names:
        barcode = bc.split(":")[-1]
        # Strip trailing 'x', append batch suffix
        barcode = barcode.rstrip("x") + "-" + suffix
        new_names.append(barcode)
    adata.obs_names = new_names
    return adata

In [25]:
for tp, adata in looms.items():
    looms[tp] = harmonize_loom_barcodes(adata, tp)
    print(f"{tp} harmonized: {looms[tp].obs_names[:3].tolist()}")

day0_2 harmonized: ['AAACGCGCAAGCTTAT-1-1', 'AAACCGAAGTCATGCG-1-1', 'AAACATGCAGCTCAAC-1-1']
day3_4 harmonized: ['AAACCAACACCTGCCT-1-2', 'AAACATGCATAATCCG-1-2', 'AAACCGCGTTTCCTCC-1-2']
day5_6 harmonized: ['AAAGCCGCACCTCACC-1-3', 'AAACCGCGTGAGACTC-1-3', 'AAACGTACATGAGTTT-1-3']


In [26]:
# Concatenate all timepoints
adata_rna = ad.concat(list(looms.values()), join='outer')
adata_rna.var_names_make_unique()
print(f"\nCombined loom: {adata_rna.shape}")
print(f"Layers: {list(adata_rna.layers.keys())}")


Combined loom: (36306, 36601)
Layers: ['spliced', 'unspliced', 'ambiguous']


In [27]:
# Check for duplicate barcodes (would indicate samples share barcodes)
n_unique = adata_rna.obs_names.nunique()
print(f"Unique barcodes: {n_unique} / {adata_rna.n_obs}")
if n_unique < adata_rna.n_obs:
    print("⚠️  DUPLICATE BARCODES DETECTED — you need sample prefixes (use OPTION B or C above)")

Unique barcodes: 36306 / 36306


In [30]:
# %%
atac_samples = {}
for tp, outs_dir in CELLRANGER_ARC_OUTS.items():
    print(f"\n--- Processing ATAC for {tp} ---")
    
    mtx_dir = os.path.join(outs_dir, "filtered_feature_bc_matrix")
    peak_annot = os.path.join(outs_dir, "atac_peak_annotation.tsv")
    feat_link = os.path.join(outs_dir, "feature_linkage.bedpe")
    
    # Check files exist
    for f in [mtx_dir, peak_annot, feat_link]:
        exists = os.path.exists(f)
        print(f"  {'✓' if exists else '✗'} {f}")
    
    # Load full feature matrix (RNA + ATAC), then subset to peaks
    adata_full = sc.read_10x_mtx(mtx_dir, var_names='gene_symbols', cache=False, gex_only=False)
    adata_peaks = adata_full[:, adata_full.var['feature_types'] == "Peaks"].copy()
    print(f"  Raw peaks: {adata_peaks.shape}")
    
    # Aggregate peaks to gene level
    import scipy.sparse
    scipy.sparse.csc_matrix.A = property(lambda self: self.toarray())
    scipy.sparse.csr_matrix.A = property(lambda self: self.toarray())
    
    adata_peaks = mv.aggregate_peaks_10x(adata_peaks, peak_annot, feat_link)
    print(f"  Gene-aggregated ATAC: {adata_peaks.shape}")
    
    adata_peaks.obs['timepoint'] = tp
    atac_samples[tp] = adata_peaks


--- Processing ATAC for day0_2 ---
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day0_2/outs/filtered_feature_bc_matrix
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day0_2/outs/atac_peak_annotation.tsv
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day0_2/outs/feature_linkage.bedpe
  Raw peaks: (15285, 155769)


  0%|          | 0/20445 [00:00<?, ?it/s]

  Gene-aggregated ATAC: (15285, 20445)

--- Processing ATAC for day3_4 ---
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day3_4/outs/filtered_feature_bc_matrix
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day3_4/outs/atac_peak_annotation.tsv
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day3_4/outs/feature_linkage.bedpe
  Raw peaks: (11127, 169355)


  0%|          | 0/20610 [00:00<?, ?it/s]

  Gene-aggregated ATAC: (11127, 20610)

--- Processing ATAC for day5_6 ---
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day5_6/outs/filtered_feature_bc_matrix
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day5_6/outs/atac_peak_annotation.tsv
  ✓ /work/hdd/bgdb/asachan/datasets/B_cell_dictys_actb1_added_v2/velocity_related/velocyto_day5_6/outs/feature_linkage.bedpe
  Raw peaks: (9894, 168188)


  0%|          | 0/20598 [00:00<?, ?it/s]

  Gene-aggregated ATAC: (9894, 20598)


In [31]:
# Concatenate
adata_atac = ad.concat(list(atac_samples.values()), join='outer')
adata_atac.var_names_make_unique()
print(f"\nCombined ATAC: {adata_atac.shape}")


Combined ATAC: (36306, 21902)


In [32]:
# Find shared cells across all three objects
shared_cells_rna_atac = pd.Index(np.intersect1d(adata_rna.obs_names, adata_atac.obs_names))
shared_cells_all = pd.Index(np.intersect1d(shared_cells_rna_atac, adata_ref.obs_names))
 
print(f"Loom (RNA) cells:        {adata_rna.n_obs}")
print(f"ATAC cells:              {adata_atac.n_obs}")
print(f"Reference h5ad cells:    {adata_ref.n_obs}")
print(f"RNA ∩ ATAC:              {len(shared_cells_rna_atac)}")
print(f"RNA ∩ ATAC ∩ Reference:  {len(shared_cells_all)}")
 
if len(shared_cells_all) == 0:
    print("\n⚠️  ZERO OVERLAP — barcodes don't match. Print examples and fix harmonize_loom_barcodes():")
    print(f"  Loom:      {adata_rna.obs_names[:3].tolist()}")
    print(f"  ATAC:      {adata_atac.obs_names[:3].tolist()}")
    print(f"  Reference: {adata_ref.obs_names[:3].tolist()}")

Loom (RNA) cells:        36306
ATAC cells:              36306
Reference h5ad cells:    28494
RNA ∩ ATAC:              0
RNA ∩ ATAC ∩ Reference:  0

⚠️  ZERO OVERLAP — barcodes don't match. Print examples and fix harmonize_loom_barcodes():
  Loom:      ['AAACGCGCAAGCTTAT-1-1', 'AAACCGAAGTCATGCG-1-1', 'AAACATGCAGCTCAAC-1-1']
  ATAC:      ['AAACAGCCAAACCTTG-1', 'AAACAGCCAAAGCTAA-1', 'AAACAGCCAAGGTGCA-1']
  Reference: ['AAACAGCCAAGCCACT-3', 'AAACAGCCAAGGTGCA-1', 'AAACAGCCAAGTTATC-1']
